# Data Mining Assignment — Spark Data Lake (Bronze & Silver)

**Student:** Sameer Singh  
**Roll number:** MSD25008


## 1. Install PySpark and start Spark

In [1]:
!pip -q install pyspark==3.5.3

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.3 which is incompatible.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os
import shutil
import glob
import urllib.request

spark = (SparkSession.builder
         .appName("DataMiningSilverLayer")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .getOrCreate())
print("Spark version:", spark.version)

Spark version: 3.5.3


## 2. Create Bronze and Silver folders, then keep the original file unchanged

In [3]:
os.makedirs("/content/data_lake/bronze", exist_ok=True)
os.makedirs("/content/data_lake/silver", exist_ok=True)

source_file = "/content/sales.csv"
if not os.path.isfile(source_file):
    url = "https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv"
    urllib.request.urlretrieve(url, source_file)

bronze_file = "/content/data_lake/bronze/sales.csv"
shutil.copyfile(source_file, bronze_file)

print("Original dataset copied to:", bronze_file)
print("Bronze file size:", os.path.getsize(bronze_file), "bytes")

Original dataset copied to: /content/data_lake/bronze/sales.csv
Bronze file size: 109537 bytes


## 3. Read Bronze and inspect the dataset

I am reading all columns as strings first, so bad numbers and dates will not disappear before I check them.

In [4]:
df = spark.read.option("header", True).option("inferSchema", False).csv(bronze_file)
print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))
df.printSchema()
df.show(8, truncate=False)

Number of rows: 1000
Number of columns: 13
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product      |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+-------------+---------+--------+----

## 4. Check exact duplicates → remove them

First I count duplicate **whole rows**. I do not remove rows just because an `order_id` is repeated: the assignment asks for *exact* duplicates.

In [5]:
before_count = df.count()
duplicate_count = before_count - df.dropDuplicates().count()
print("Rows before:", before_count)
print("Exact duplicate rows:", duplicate_count)

df = df.dropDuplicates()
print("Rows after removing exact duplicates:", df.count())

Rows before: 1000
Exact duplicate rows: 10
Rows after removing exact duplicates: 990


## 5. Check missing values → fill only the obvious text fields

First I count blanks (including strings that contain only spaces). Then I use `Unknown` for missing `customer_name`, `payment_method`, and `city`. This keeps useful sales records instead of throwing away complete rows.

In [6]:
print("Missing / blank values in each column:")
missing_before = df.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in df.columns
])
missing_before.show(truncate=False)

for c in ["customer_name", "payment_method", "city"]:
    df = df.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), "Unknown")
                       .otherwise(F.col(c)))

print("Missing values after filling selected text columns:")
df.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in ["customer_name", "payment_method", "city"]
]).show()

Missing / blank values in each column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

Missing values after filling selected text columns:
+-------------+--------------+----+
|customer_name|payment_method|city|
+-------------+--------------+----+
|            0|             0|   0|
+-------------+--------------+----+



## 6. Check extra spaces → trim text

I compare each text value against its trimmed version. If a column has differences, it has leading or trailing spaces. Then I trim all text columns and show the remaining count.

In [7]:
print("Cells with leading/trailing spaces BEFORE:")
for c in df.columns:
    n = df.filter(F.col(c).isNotNull() & (F.col(c) != F.trim(F.col(c)))).count()
    if n:
        print(c, "->", n)
        df.filter(F.col(c) != F.trim(F.col(c))).select(c).show(5, truncate=False)

for c in df.columns:
    df = df.withColumn(c, F.trim(F.col(c)))

spaces_after = sum(df.filter(F.col(c).isNotNull() & (F.col(c) != F.trim(F.col(c)))).count() for c in df.columns)
print("Cells with extra outer spaces AFTER:", spaces_after)

Cells with leading/trailing spaces BEFORE:
customer_name -> 3
+----------------+
|customer_name   |
+----------------+
|Tanvi Chaudhary |
|Meera Joshi     |
|Akash Jain      |
+----------------+

city -> 3
+-----------+
|city       |
+-----------+
| Hyderabad |
| Hyderabad |
| Mumbai    |
+-----------+

Cells with extra outer spaces AFTER: 0


## 7. Check inconsistent capitalization → standardize category and city

First I look at different values. Some categories/cities describe the same thing with different capital letters. I use title case for these two fields and check their unique values again. I leave product names, customer names and acronyms such as `UPI` untouched.

In [8]:
print("Category values BEFORE:")
df.groupBy("category").count().orderBy("category").show(30, truncate=False)
print("City values BEFORE:")
df.groupBy("city").count().orderBy("city").show(30, truncate=False)

df = df.withColumn("category", F.initcap(F.lower(F.col("category"))))
df = df.withColumn("city", F.initcap(F.lower(F.col("city"))))

print("Category values AFTER:")
df.groupBy("category").count().orderBy("category").show(30, truncate=False)
print("City values AFTER:")
df.groupBy("city").count().orderBy("city").show(30, truncate=False)

Category values BEFORE:
+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |135  |
|Books          |127  |
|CLOTHING       |1    |
|Clothing       |107  |
|ELECTRONICS    |1    |
|Electronics    |103  |
|Furniture      |129  |
|Grocery        |134  |
|Home Appliances|113  |
|SPORTS         |1    |
|Sports         |136  |
|beauty         |2    |
|sports         |1    |
+---------------+-----+

City values BEFORE:
+----------+-----+
|city      |count|
+----------+-----+
|Bengaluru |90   |
|Chennai   |88   |
|Coimbatore|80   |
|DELHI     |2    |
|Delhi     |80   |
|HYDERABAD |1    |
|Hyderabad |96   |
|JAIPUR    |1    |
|Jaipur    |82   |
|Kolkata   |81   |
|Lucknow   |89   |
|Mumbai    |105  |
|Pune      |81   |
|Unknown   |8    |
|Vijayawada|104  |
|coimbatore|1    |
|delhi     |1    |
+----------+-----+

Category values AFTER:
+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |137  |
|Books          |127  |
|Clo

## 8. Check quantity → handle invalid values

Quantity should be greater than zero. I show the invalid records first. Instead of guessing a replacement quantity, I set invalid values to `null` so that the record is still available.

In [9]:
quantity_number = F.col("quantity").cast("int")
bad_quantity = df.filter(quantity_number.isNull() | (quantity_number <= 0))
print("Invalid quantity records:", bad_quantity.count())
bad_quantity.select("order_id", "quantity").show(15, truncate=False)

df = df.withColumn("quantity", F.when(quantity_number > 0, quantity_number)
                   .otherwise(F.lit(None).cast("int")))
print("Invalid quantities remaining (non-null and <= 0):", df.filter(F.col("quantity") <= 0).count())

Invalid quantity records: 12
+--------+--------+
|order_id|quantity|
+--------+--------+
|100365  |0       |
|100458  |-2      |
|100780  |0       |
|100919  |-2      |
|100114  |0       |
|100593  |-1      |
|100501  |0       |
|100198  |-2      |
|100926  |-2      |
|100143  |0       |
|100864  |-2      |
|100375  |0       |
+--------+--------+

Invalid quantities remaining (non-null and <= 0): 0


## 9. Check unit price → handle invalid values

Price should be greater than zero. I inspect invalid prices and replace them with `null` rather than inventing a price.

In [10]:
price_number = F.col("unit_price").cast("double")
bad_price = df.filter(price_number.isNull() | (price_number <= 0))
print("Invalid unit prices:", bad_price.count())
bad_price.select("order_id", "unit_price").show(15, truncate=False)

df = df.withColumn("unit_price", F.when(price_number > 0, price_number)
                   .otherwise(F.lit(None).cast("double")))
print("Invalid prices remaining (non-null and <= 0):", df.filter(F.col("unit_price") <= 0).count())

Invalid unit prices: 8
+--------+----------+
|order_id|unit_price|
+--------+----------+
|100284  |-500      |
|100102  |0         |
|100235  |-500      |
|100629  |0         |
|100816  |0         |
|100487  |-500      |
|100170  |0         |
|100930  |-250      |
+--------+----------+

Invalid prices remaining (non-null and <= 0): 0


## 10. Check discounts → handle invalid values

A percentage discount must be between 0 and 100 (inclusive). I check the bad values and change only those values to `null`.

In [11]:
discount_number = F.col("discount_percent").cast("double")
bad_discount = df.filter(discount_number.isNull() | (discount_number < 0) | (discount_number > 100))
print("Invalid discounts:", bad_discount.count())
bad_discount.select("order_id", "discount_percent").show(15, truncate=False)

df = df.withColumn("discount_percent",
                   F.when((discount_number >= 0) & (discount_number <= 100), discount_number)
                   .otherwise(F.lit(None).cast("double")))
print("Invalid non-null discounts remaining:",
      df.filter((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).count())

Invalid discounts: 8
+--------+----------------+
|order_id|discount_percent|
+--------+----------------+
|100217  |-5              |
|100034  |-5              |
|100140  |-5              |
|100310  |120             |
|100363  |120             |
|100857  |120             |
|100574  |120             |
|100999  |-5              |
+--------+----------------+

Invalid non-null discounts remaining: 0


## 11. Check date formats → convert valid dates to YYYY-MM-DD

The dataset includes different date formats and a few invalid values. First I show which dates cannot be parsed. Then I try known formats and keep an unparseable date as `null`. I never make up a new date.

In [12]:
parsed_date = F.coalesce(
    F.to_date("order_date", "yyyy-MM-dd"),
    F.to_date("order_date", "yyyy/MM/dd"),
    F.to_date("order_date", "MM-dd-yyyy"),
    F.to_date("order_date", "dd/MM/yyyy")
)
print("Dates that cannot be parsed:")
df.filter(parsed_date.isNull()).select("order_id", "order_date").show(20, truncate=False)
print("Number of unparseable dates:", df.filter(parsed_date.isNull()).count())

df = df.withColumn("order_date", F.date_format(parsed_date, "yyyy-MM-dd"))
print("Dates after standardizing (sample):")
df.select("order_id", "order_date").show(10, truncate=False)

Dates that cannot be parsed:
+--------+----------+
|order_id|order_date|
+--------+----------+
|100448  |not-a-date|
|100097  |31/02/2026|
|100637  |2026-13-05|
|100634  |2025-02-30|
+--------+----------+

Number of unparseable dates: 4
Dates after standardizing (sample):
+--------+----------+
|order_id|order_date|
+--------+----------+
|100119  |2025-07-15|
|100122  |2026-06-28|
|100177  |2026-01-18|
|100325  |2025-12-21|
|100728  |2025-06-27|
|100002  |2025-08-03|
|100150  |2025-06-15|
|100405  |2025-02-20|
|100569  |2026-01-11|
|100760  |2025-11-19|
+--------+----------+
only showing top 10 rows



## 12. Final Silver checks and counts

All counts below are calculated by Spark. We preserve rows with an unknown or invalid individual field when the rest of the sale is useful.

In [13]:
after_count = df.count()
print("Bronze rows before preprocessing:", before_count)
print("Silver rows after preprocessing:", after_count)
print("Rows removed (exact duplicates):", before_count - after_count)
print("Rows with a null in one of the validated fields:")
for c in ["quantity", "unit_price", "discount_percent", "order_date"]:
    print(c, "->", df.filter(F.col(c).isNull()).count())
print("Final Silver schema:")
df.printSchema()
df.show(8, truncate=False)

Bronze rows before preprocessing: 1000
Silver rows after preprocessing: 990
Rows removed (exact duplicates): 10
Rows with a null in one of the validated fields:
quantity -> 12
unit_price -> 8
discount_percent -> 8
order_date -> 4
Final Silver schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+-------------+-------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|custome

## 13. Save Silver output and create the required single CSV

Spark normally writes a folder containing `part-*.csv`. I save that folder as the Silver layer, then copy its CSV part as `sales_silver.csv` for the GitHub submission. `coalesce(1)` is fine for this small assignment dataset.

In [14]:
silver_folder = "/content/data_lake/silver/preprocessed_sales"
(df.coalesce(1).write.mode("overwrite").option("header", True).csv(silver_folder))

part_files = glob.glob(os.path.join(silver_folder, "part-*.csv"))
assert len(part_files) == 1, "Expected one CSV part file"
submission_csv = "/content/sales_silver.csv"
shutil.copyfile(part_files[0], submission_csv)
print("Silver folder:", silver_folder)
print("GitHub submission CSV:", submission_csv)

Silver folder: /content/data_lake/silver/preprocessed_sales
GitHub submission CSV: /content/sales_silver.csv


## 14. Read the Silver folder back using Spark and verify

This is required to show that the saved Silver output is readable.

In [15]:
silver_check = spark.read.option("header", True).option("inferSchema", False).csv(silver_folder)
print("Saved Silver row count:", silver_check.count())
silver_check.show(8, truncate=False)
assert silver_check.count() == after_count, "Saved Silver row count does not match!"
print("Verification successful!")

Saved Silver row count: 990
+--------+-----------+-------------+-------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name|product      |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+-------------+-------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100119  |C0191      |Aman Naidu   |Cricket Bat  |Sports         |5       |2450.0    |12.0            |Net Banking     |Mumbai    |Maharashtra   |2025-07-15|Delivered   |
|100122  |C0188      |Ananya Sharma|Sugar 5kg    |Grocery        |6       |340.0     |10.0            |Debit Card      |Vijayawada|Andhra Pradesh|2026-06-28|Shipped     |
|100177  |C0072      |Arjun Sharma |Ceiling Fan  |Home Appliances|7       |3920.0    |30.0            |Cash on Delive

## 16. Download the Silver CSV in Colab

Run this cell to download the Silver CSV to your computer. Download the notebook separately with **File → Download → Download .ipynb**.

In [16]:
from google.colab import files
files.download("/content/sales_silver.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>